[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/14-deployment-and-llmops/notebooks/02-docker-uis-and-serving.ipynb)

# 02 · Docker, quick UIs and model serving — understood offline

Docker and vLLM cannot run inside this notebook (they need a Docker engine or a GPU). Instead we **simulate the ideas** so you understand exactly what happens when you do run them:

1. Read a Dockerfile as data and see which **layers** rebuild after a change — good vs bad ordering.
2. Read a `docker-compose.yml` and list services, ports and dependencies.
3. Build a Gradio chat UI object and call its function directly (no server).
4. Simulate **static vs continuous batching** and **PagedAttention** memory use, the two big ideas behind vLLM.

**Words used below**
- **Image** — a frozen package of an app and everything it needs. **Container** — a running copy of an image.
- **Layer** — the saved result of one Dockerfile instruction; unchanged layers are reused from the **cache**.
- **Batch** — several requests processed by the GPU together.

## Step 0 — is Docker installed here?

`shutil.which` looks for a program on this computer. In Colab and in this course's build machine there is usually no Docker engine, so we simulate. On your laptop, install Docker Desktop (Windows/macOS) or Docker Engine (Linux) and run the commands from `concepts.md`.

In [1]:
import shutil
print("docker CLI found at:", shutil.which("docker"))
print("We will simulate Docker's behaviour in Python either way.")

docker CLI found at: /usr/bin/docker
We will simulate Docker's behaviour in Python either way.


## Step 1 — a Dockerfile is a list of instructions

Here is the Dockerfile from `code/Dockerfile` (comments removed). We split it into `(INSTRUCTION, argument)` pairs. Each pair becomes one **layer**.

In [2]:
GOOD_DOCKERFILE = """
FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app/
USER appuser
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

def parse(dockerfile: str):
    steps = []
    for line in dockerfile.strip().splitlines():
        instr, _, arg = line.partition(" ")
        steps.append((instr, arg))
    return steps

for i, (instr, arg) in enumerate(parse(GOOD_DOCKERFILE), 1):
    print(f"layer {i}: {instr:8s} {arg}")

layer 1: FROM     python:3.13-slim
layer 2: WORKDIR  /app
layer 3: COPY     requirements.txt .
layer 4: RUN      pip install --no-cache-dir -r requirements.txt
layer 5: COPY     app/ ./app/
layer 6: USER     appuser
layer 7: CMD      ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


## Step 2 — simulate the build cache

Docker's rule (simplified): a layer is reused if **its instruction and the files it copies are unchanged, and every layer before it was reused**. As soon as one layer changes, it and *all later layers* rebuild.

We give each file a pretend "content version". `COPY` steps depend on the files they copy; `RUN pip install` is slow (we say 60 s), the others are fast.

In [3]:
def build(dockerfile: str, files: dict, cache: dict):
    """Return (total_seconds, log) and update the cache."""
    total, log, still_cached = 0, [], True
    for i, (instr, arg) in enumerate(parse(dockerfile)):
        src = arg.split()[0] if instr == "COPY" else None
        if src == ".":
            inputs = tuple(sorted(files.items()))           # COPY . . depends on EVERY file
        elif src:
            inputs = tuple((k, v) for k, v in files.items() if k.startswith(src.rstrip("/")))
        else:
            inputs = ()
        key = (i, instr, arg, inputs)
        if still_cached and cache.get(i) == key:
            log.append(f"  CACHED  {instr} {arg[:40]}")
        else:
            still_cached = False
            secs = 60 if "pip install" in arg else 1
            total += secs
            log.append(f"  BUILD   {instr} {arg[:40]}  ({secs}s)")
            cache[i] = key
    return total, log

files = {"requirements.txt": "v1", "app/main.py": "v1"}
cache = {}
t, log = build(GOOD_DOCKERFILE, files, cache); print(f"First build: {t}s"); print("\n".join(log))
files["app/main.py"] = "v2"                                 # edit our code
t, log = build(GOOD_DOCKERFILE, files, cache); print(f"\nAfter editing main.py: {t}s"); print("\n".join(log))

First build: 66s
  BUILD   FROM python:3.13-slim  (1s)
  BUILD   WORKDIR /app  (1s)
  BUILD   COPY requirements.txt .  (1s)
  BUILD   RUN pip install --no-cache-dir -r requiremen  (60s)
  BUILD   COPY app/ ./app/  (1s)
  BUILD   USER appuser  (1s)
  BUILD   CMD ["uvicorn", "app.main:app", "--host", "0  (1s)

After editing main.py: 3s
  CACHED  FROM python:3.13-slim
  CACHED  WORKDIR /app
  CACHED  COPY requirements.txt .
  CACHED  RUN pip install --no-cache-dir -r requiremen
  BUILD   COPY app/ ./app/  (1s)
  BUILD   USER appuser  (1s)
  BUILD   CMD ["uvicorn", "app.main:app", "--host", "0  (1s)


## Step 3 — the same change with a badly ordered Dockerfile

A very common beginner Dockerfile copies *everything* first (`COPY . .`) and installs afterwards. Now any code edit invalidates the copy layer, so the slow `pip install` runs again every time.

In [4]:
BAD_DOCKERFILE = """
FROM python:3.13-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
files = {"requirements.txt": "v1", "app/main.py": "v1"}
cache = {}
build(BAD_DOCKERFILE, files, cache)
files["app/main.py"] = "v2"
t, log = build(BAD_DOCKERFILE, files, cache)
print(f"Bad order, after editing main.py: {t}s"); print("\n".join(log))

Bad order, after editing main.py: 62s
  CACHED  FROM python:3.13-slim
  CACHED  WORKDIR /app
  BUILD   COPY . .  (1s)
  BUILD   RUN pip install --no-cache-dir -r requiremen  (60s)
  BUILD   CMD ["uvicorn", "app.main:app", "--host", "0  (1s)


**Lesson:** put things that change *rarely* (base image, requirements) at the top and things that change *often* (your code) at the bottom. Ten edits a day × 60 s saved = real time.

## Step 4 — reading docker-compose.yml

**Docker Compose** starts several containers from one **YAML** file (YAML = settings written with indentation). We parse a copy of our compose file with the `yaml` library and print what each service does. Notice `API_URL: http://api:8000` — inside Compose, a service's *name* works as its network address.

In [5]:
import yaml   # PyYAML; preinstalled in Colab

COMPOSE = """
services:
  api:
    build: .
    ports: ["8000:8000"]
    env_file: [.env]
    environment: {APP_ENV: docker}
  ui:
    image: python:3.13-slim
    ports: ["8501:8501"]
    environment: {API_URL: "http://api:8000"}
    depends_on: [api]
"""
spec = yaml.safe_load(COMPOSE)
for name, svc in spec["services"].items():
    source = "built from Dockerfile" if "build" in svc else f"image {svc['image']}"
    for mapping in svc.get("ports", []):
        host, container = mapping.split(":")
        print(f"{name:4s}: {source:22s} | laptop port {host} -> container port {container}"
              f" | depends on {svc.get('depends_on', [])}")

api : built from Dockerfile  | laptop port 8000 -> container port 8000 | depends on []
ui  : image python:3.13-slim | laptop port 8501 -> container port 8501 | depends on ['api']


## Step 5 — a Gradio chat UI without starting a server

**Gradio** turns a Python function into a web page. We define `respond(message, history)` and wrap it in `gr.ChatInterface`. We do **not** call `.launch()` here (that would start a web server and block the notebook); instead we call `respond` directly — exactly what Gradio does when a user presses Enter.

If Gradio is not installed (it is optional), the cell explains how to get it and still tests the function.

In [6]:
def respond(message: str, history: list) -> str:
    """The function Gradio would call. Offline stand-in for the API call."""
    return f"(offline demo) You said: {message}"

try:
    import gradio as gr
    demo = gr.ChatInterface(fn=respond, title="Module 14 demo")
    print("Gradio", gr.__version__, "UI object built:", type(demo).__name__)
    print("To show it in Colab or locally run: demo.launch()")
except ImportError:
    print("Gradio not installed. Install with:  pip install gradio  (then demo.launch())")

print(respond("What is a container?", []))

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.32.0 UI object built: ChatInterface
To show it in Colab or locally run: demo.launch()
(offline demo) You said: What is a container?


## Step 6 — Streamlit, tested headlessly

**Streamlit** re-runs the whole script on every click. Its `AppTest` tool runs a Streamlit script *without a browser*, which is how you test Streamlit apps in CI. We write a tiny app to a temp file and test it. (Skipped politely if Streamlit is not installed — it is optional.)

In [7]:
import tempfile, pathlib, textwrap
APP = textwrap.dedent("""
    import streamlit as st
    st.title("Echo bot")
    if "n" not in st.session_state:
        st.session_state.n = 0
    msg = st.text_input("Message")
    if msg:
        st.session_state.n += 1
        st.write(f"You said: {msg} (message #{st.session_state.n})")
""")
try:
    from streamlit.testing.v1 import AppTest
    path = pathlib.Path(tempfile.mkdtemp()) / "echo.py"
    path.write_text(APP)
    at = AppTest.from_file(str(path)).run()
    at.text_input[0].input("hello").run()          # type into the box -> script re-runs
    print("Rendered:", at.markdown[0].value)
except ImportError:
    print("Streamlit not installed. Install with: pip install streamlit ; run with: streamlit run app.py")

2026-11-21 22:10:11.715 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Rendered: You said: hello (message #1)


## Step 7 — vLLM idea #1: static vs continuous batching

A GPU generates one token per step for every request in its **batch**. We simulate 4 GPU slots and 12 requests with different answer lengths.

- **Static batching**: take 4 requests, run until **all 4** finish, then take the next 4. Short requests leave their slot idle.
- **Continuous batching**: after **every step**, a finished request leaves and a waiting one joins immediately.

We count total steps and the **utilisation** (share of slot-steps doing useful work).

In [8]:
import numpy as np
rng = np.random.default_rng(7)
lengths = [int(x) for x in rng.integers(5, 60, size=12)]   # tokens each request needs
SLOTS = 4
print("answer lengths:", lengths)

def static_batching(lengths, slots):
    timeline, t = [], 0
    for b in range(0, len(lengths), slots):
        batch = lengths[b:b + slots]
        longest = max(batch)
        for j, L in enumerate(batch):
            timeline.append((j, t, L, b + j))       # (slot, start, length, request id)
        t += longest                                # everyone waits for the longest
    return timeline, t

def continuous_batching(lengths, slots):
    free_at = [0] * slots                           # when each slot becomes free
    timeline = []
    for rid, L in enumerate(lengths):
        j = int(np.argmin(free_at))                 # the first slot to free up takes the next request
        timeline.append((j, free_at[j], L, rid))
        free_at[j] += L
    return timeline, max(free_at)

st_tl, st_T = static_batching(lengths, SLOTS)
co_tl, co_T = continuous_batching(lengths, SLOTS)
useful = sum(lengths)
print(f"static    : {st_T} steps, utilisation {useful / (st_T * SLOTS):.0%}")
print(f"continuous: {co_T} steps, utilisation {useful / (co_T * SLOTS):.0%}")

answer lengths: [56, 39, 42, 54, 36, 47, 50, 17, 8, 21, 20, 53]
static    : 159 steps, utilisation 70%
continuous: 142 steps, utilisation 78%


## Step 8 — draw both timelines

Each row is a GPU slot; each coloured bar is one request; gaps are idle slots.

In [9]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
cmap = plt.get_cmap("tab20")
for ax, (tl, T, title) in zip(axes, [(st_tl, st_T, "Static batching"), (co_tl, co_T, "Continuous batching")]):
    ax.add_patch(plt.Rectangle((0, -0.5), T, SLOTS, color="#e5e7eb"))      # grey = idle
    for slot, start, L, rid in tl:
        ax.barh(slot, L, left=start, color=cmap(rid % 20), edgecolor="white")
    ax.set_title(f"{title}: {T} steps"); ax.set_xlabel("token steps"); ax.set_xlim(0, max(st_T, co_T))
axes[0].set_ylabel("GPU slot"); axes[0].set_yticks(range(SLOTS))
plt.tight_layout(); plt.show()

## Step 9 — vLLM idea #2: PagedAttention and memory waste

Each request needs GPU memory for its **KV cache** (saved numbers for every token so far). Old servers reserved space for the **maximum** length up front. **PagedAttention** hands out small **blocks** (here 16 tokens each) only as a request grows.

We compare how much reserved memory is actually used, and how many requests fit in a fixed memory budget.

In [10]:
MAX_LEN, BLOCK, BUDGET = 2048, 16, 40_000          # tokens of KV-cache space on our pretend GPU
actual = rng.integers(50, 600, size=200)           # real lengths of 200 requests (most are short)

reserved_old = MAX_LEN                              # per request, reserved up front
reserved_paged = np.ceil(actual / BLOCK) * BLOCK    # rounded up to whole blocks

print(f"old way   : uses {actual.mean() / reserved_old:.0%} of what it reserves; "
      f"fits {BUDGET // reserved_old} requests at once")
fit = int(np.searchsorted(np.cumsum(reserved_paged), BUDGET))
print(f"paged way : uses {actual.sum() / reserved_paged.sum():.0%} of what it reserves; "
      f"fits about {fit} requests at once")

old way   : uses 16% of what it reserves; fits 19 requests at once
paged way : uses 98% of what it reserves; fits about 116 requests at once


## Step 10 — what the real vLLM call looks like

With a GPU you would start: `vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8001 --api-key $VLLM_API_KEY` and then call it in the **OpenAI-compatible** format. The cell below sends the request only if `VLLM_BASE_URL` is set; otherwise it prints the JSON body it would send. The same body works for OpenAI, Ollama, LM Studio and others — only the address changes.

In [11]:
import os, json
payload = {"model": "Qwen/Qwen2.5-0.5B-Instruct",
           "messages": [{"role": "user", "content": "What is continuous batching?"}],
           "max_tokens": 60}
base = os.getenv("VLLM_BASE_URL")
if base:
    import httpx
    r = httpx.post(f"{base}/chat/completions", json=payload,
                   headers={"Authorization": f"Bearer {os.getenv('VLLM_API_KEY', '')}"}, timeout=60)
    print(r.json()["choices"][0]["message"]["content"])
else:
    print("No VLLM_BASE_URL set. Request body that would be POSTed to <base>/chat/completions:")
    print(json.dumps(payload, indent=2))

No VLLM_BASE_URL set. Request body that would be POSTed to <base>/chat/completions:
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "messages": [
    {
      "role": "user",
      "content": "What is continuous batching?"
    }
  ],
  "max_tokens": 60
}


## Recap

- Order your Dockerfile from rarely-changing to often-changing to keep builds fast.
- Compose wires services together; service names are hostnames.
- Gradio = function → page; Streamlit = script that re-runs. Both should call your API, not hold keys.
- vLLM is fast because **continuous batching** keeps GPU slots busy and **PagedAttention** wastes far less memory, so more requests fit at once.

**Try it yourself:** change `SLOTS` to 8 and `lengths` to include one very long request (say 300). Which strategy suffers more?